In [0]:
#objetivo dessa camada: unificar dois cadastros do mesmo assunto
#camada prata é onde fazemos parte de governança, auditoria, etc
#aqui garantimos que o dado tá com qualidade, tá como nós esperamos pra gente efetivamente conseguir fazer as análises.

#permitido na prata:

#tipagem | string vira `TIMESTAMP`, `INT`, `DATE` | 
#legibilidade | quebrar timestamp em data e hora | 
#metadados | `COMMENT` em toda coluna, tags na tabela | --> metadado é o dado sobre o dado --> dar contexto pra IA, só o dado solto ela pode alucinar. Dar contexto sobre o dado, ou seja, dado sobre o dado, é importante.
#unificação | dois cadastros do mesmo assunto, com a origem por registro | 
#aritmética pura | `atraso = real - previsto` | --> ao invés de fazer esse mesmo cálculo para cada análise na ouro, faço de uma vez na prata, sem alterar nenhuma coluna, só usando os dados que já temos e gerando uma nova. 

#não permitido na prata: 

#filtro / `WHERE` de negócio |
#`GROUP BY` / agregação | 
#limiar, flag, classificação |

#sem eliminação de dados. Transformando dados da bronze em dados com governança. Estamos garantindo a qualidade do dado. Eliminação de dados por conta de regras de negócio só na camada ouro. Objetivo aqui é deixar dados prontos para que eu possa consumi-los na ouro pra fazer análises de negócios.
#Na bronze, objetivos é deixar dados presentes, pegar os dados como sao, colocar no ambiente. Na silver, dar qualidade e capacidade pra eles serem consumidos para aspectos de negócios.

#depois da ingestão, temos que fazer uma análise exploratória. Ver problemas na origem e na tabela bronze sobre nulidade, sobre strings estranhas, etc.

In [0]:
#começando, vamos olhar para algumas colunas que podem ter dados nulos. Iniciando a analise exploratória
display(spark.sql("""
    SELECT
      COUNT(*)                                                        AS linhas,
      SUM(CASE WHEN partida_real     IS NULL THEN 1 ELSE 0 END)       AS partida_real_null_de_verdade,
      SUM(CASE WHEN partida_real     = 'null' THEN 1 ELSE 0 END)      AS partida_real_string_null,
      SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END)      AS partida_prevista_string_null,
      SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END)    AS com_fracao_de_segundo
    FROM voebem.bronze.vra
"""))
#aqui em cima é que, se tiver dado nulo, ele retornará 0
#Isso não é regra de negócio. Não podemos ter dados nulos, isso é princípio básico, questão de qualidade dos dados.
# no fim, vai mostrar quantas linhas estão nulas em algumas colunas da nossa tabela vra

In [0]:
%sql

--precisamos criar um schema da camada prata
CREATE SCHEMA IF NOT EXISTS voebem.silver

In [0]:
#query em algumas colunas que queremos. O try_cast ele vai tentar transformar o que for nulo das colunas em timestamp, porque vão ser usados pra cálculo então não queremos o dado nulo.


#Armadilha 1 — A string 'null' (4 caracteres): WHERE partida_real IS NULL devolve zero numa tabela onde 29 mil voos não têm horário real. Correção: nullif(coluna, 'null') antes do cast. 

# Armadilha 2 — Múltiplos formatos de timestamp no mesmo arquivo: A maioria das linhas segue o padrão 2026-01-27 19:45:00, mas ~80 mil linhas vêm com fração de segundo de 9 casas. Um to_timestamp(col, 'yyyy-MM-dd HH:mm:ss') fixo devolveria NULL silenciosamente para 8% da base. O try_cast(... AS TIMESTAMP) aceita ambos os formatos, e o prefixo try_ garante que novos formatos virem NULL em vez de derrubar o job.

#Nota: Trata-se de tipagem, não limpeza de negócio. Traduzir 'null' para NULL apenas expressa a ausência de dado no tipo correto, sem descartar nenhuma linha.

#Repare no que não existe nesta query: nenhum WHERE, nenhum GROUP BY, nenhum DISTINCT, nenhum JOIN. É um SELECT de projeção sobre o bronze inteiro. E repare nas três colunas do fim: atraso_partida_min, atraso_chegada_min e minutos_recuperados. São subtrações entre colunas da própria linha. Não têm limiar, não classificam nada, não escondem número mágico — e, principalmente, não impedem análise nenhuma. Por isso podem morar aqui.

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.vra AS
WITH tipado AS (
  SELECT
    icao_empresa,
    numero_voo,
    codigo_di,
    codigo_tipo_linha,
    icao_origem,
    icao_destino,
    try_cast(nullif(partida_prevista, 'null') AS TIMESTAMP) AS partida_prevista,
    try_cast(nullif(partida_real,     'null') AS TIMESTAMP) AS partida_real,
    try_cast(nullif(chegada_prevista, 'null') AS TIMESTAMP) AS chegada_prevista,
    try_cast(nullif(chegada_real,     'null') AS TIMESTAMP) AS chegada_real,
    situacao_voo,
    nullif(codigo_justificativa, 'N/A')                     AS codigo_justificativa, --quando código_justificativa tiver nulo, vai ser colocado o 'N/A'
    _arquivo_origem,
    _ingerido_em
  FROM voebem.bronze.vra
)

--Segundo select usando nossa cte que construímos acima. cte foi usada pq é mais performática e mais legível. não preciso criar outra tabela ou dataframe pra seguir, cte supre isso.
SELECT
  icao_empresa,
  numero_voo,
  codigo_di,
  codigo_tipo_linha,
  icao_origem,
  icao_destino,

  partida_prevista,
  CAST(partida_prevista AS DATE)                     AS partida_prevista_data, 
  date_format(partida_prevista, 'HH:mm')             AS partida_prevista_hora, --cast quer dizer transformar dado, transformando de string para date, e o date_format mostra que é hora e minuto. tamos mudando a tipagem do dado

  partida_real,
  CAST(partida_real AS DATE)                         AS partida_real_data,
  date_format(partida_real, 'HH:mm')                 AS partida_real_hora,

  chegada_prevista,
  CAST(chegada_prevista AS DATE)                     AS chegada_prevista_data,
  date_format(chegada_prevista, 'HH:mm')             AS chegada_prevista_hora,

  chegada_real,
  CAST(chegada_real AS DATE)                         AS chegada_real_data,
  date_format(chegada_real, 'HH:mm')                 AS chegada_real_hora,

  situacao_voo,
  codigo_justificativa,

  -- aritmetica pura: subtracao de colunas da propria linha, sem limiar e sem decisao
  CAST(timestampdiff(MINUTE, partida_prevista, partida_real) AS INT) AS atraso_partida_min,
  CAST(timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS atraso_chegada_min,
  CAST(timestampdiff(MINUTE, partida_prevista, partida_real)
     - timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutos_recuperados,

  _arquivo_origem,
  _ingerido_em,
  current_timestamp()                                AS _transformado_em
FROM tipado
""")

print("silver.vra criada")

#pyspark usa sql ansi, o tipo mais primitivo e simples existente. acima, usamos CTEs que é parte do padrão ansi. CTE é uma gestao de tabela temporária. a saída do select dessa tabela vai gerar dados tipados, e esses dados vao virar uma tabela temporária
#timestampdiff e MINUTE são funções que foram pré-disponibilizadas dentro do sql pelo pyspark
#cuidado com scripts que precisam de acesso externo, ou seja, que nao rodam puramente offline! é uma brecha de segurança
#então finalizada a parte inicial da silver, onde fizemos algumas tipagens específicas, criamos algumas colunas novas pra guardar as horas e fizemos cálculos pros atrasos

In [0]:
#a intenção não é eliminar nenhum dado, mas sim normalizá-los e deixá-los pronto para uso. Então não se pode ter uma diferença na quantidade de linhas em relação à bronze. o código abaixo vai nos mostrar isso. Precisamos que a diferença dê 0
display(spark.sql("""
    SELECT
      (SELECT COUNT(*) FROM voebem.bronze.vra) AS bronze_vra,
      (SELECT COUNT(*) FROM voebem.silver.vra) AS silver_vra,
      (SELECT COUNT(*) FROM voebem.bronze.vra)
        - (SELECT COUNT(*) FROM voebem.silver.vra) AS diferenca
"""))

In [0]:
#checagem --> contagem de conversões bem-sucedidas por coluna. Passo importante para ver se tá dando certo, manter o controle e entender o dado
display(spark.sql("""
    SELECT
      COUNT(partida_prevista)     AS partida_prevista_ok,
      COUNT(partida_real)         AS partida_real_ok,
      COUNT(chegada_prevista)     AS chegada_prevista_ok,
      COUNT(chegada_real)         AS chegada_real_ok,
      COUNT(atraso_partida_min)   AS atraso_partida_ok,
      COUNT(minutos_recuperados)  AS minutos_recuperados_ok
    FROM voebem.silver.vra
"""))

In [0]:
#ordenando dados pela partida prevista, e vamos visualizar só 5 linhas
display(spark.sql("""
    SELECT icao_empresa, numero_voo, icao_origem, icao_destino,
           partida_prevista, partida_prevista_data, partida_prevista_hora,
           atraso_partida_min, atraso_chegada_min, minutos_recuperados, situacao_voo
    FROM voebem.silver.vra
    ORDER BY partida_prevista
    LIMIT 5
"""))

In [0]:
#nós temos dados vindo de duas fontes diferentes que tratam do mesmo assunto: processos administrativos da ANAC descrevendo a mesma entidade de negócio --> "empresa aérea que opera no Brasil".
#agora vamos unificar os dados
spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.empresas AS
SELECT
  icao,
  sigla_iata,
  razao_social,
  servico,
  cidade,
  uf,
  situacao,
  'nacional'      AS origem_cadastro,
  _arquivo_origem,
  _ingerido_em,
  current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_nacionais
UNION ALL --esse union que junto as duas tabelas que estamos utilizando
SELECT
  icao,
  sigla_iata,
  razao_social,
  servico,
  cidade,
  uf,
  situacao,
  'estrangeira'   AS origem_cadastro,
  _arquivo_origem,
  _ingerido_em,
  current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_estrangeiras
""")
#até agora estavamos usando as tabelas de formas separadas. Precisava visualizar as duas como duas coisas diferentes. Mas possuem os mesmos dados. Para análise, é o ideal analisarmos tudo de uma vez, sem precisar ficar fazendo pesquisa nas duas. Unimos elas e se tornaram uma tabela unica, com os dados de ambas tabelas de origem, de empresas estrangeiras e nacionais.

#na bronze, usamos criação de dataframe para popular os dados e prepará-lo pra sair. na prata, usamos cte, e agora estamos criando tabela como resultado direto de um select. A versatilidade do sql resolve mt coisa. no tradicional do pyspark, teriamos mt mais etapas
#nao criamos linhas, mas fizemos enriquecimento de tabelas na silver, para tabelas estarem prontas para analises de negocio, as vezes pegando varias bases de dados diferentes mas que possuem dados complementares

#se quiser ter uma informação especifica, é melhor ter todas elas e fazer um filtro do que só ter uma informação

#vamos garantir que estão certas. contamos quantas linhas cada tabela na bronze tinha, a soma das duas na bronze e o quanto a nossa tabela que fizemos com o union tem, pra ver se deu certinho os valores.
display(spark.sql("""
    SELECT
      (SELECT COUNT(*) FROM voebem.bronze.empresas_nacionais)    AS bronze_nacionais,
      (SELECT COUNT(*) FROM voebem.bronze.empresas_estrangeiras) AS bronze_estrangeiras,
      (SELECT COUNT(*) FROM voebem.bronze.empresas_nacionais)
        + (SELECT COUNT(*) FROM voebem.bronze.empresas_estrangeiras) AS soma_esperada,
      (SELECT COUNT(*) FROM voebem.silver.empresas)              AS silver_empresas
"""))

In [0]:
#contando linhas e vendo se o icao tá nulo. precisamos que nao fique nulo, entao estamos fazendo a alteração para que ele fique vazio ao invés de nulo
display(spark.sql("""
    SELECT origem_cadastro,
           COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao --ponto de tratamento
    FROM voebem.silver.empresas
    GROUP BY origem_cadastro --unifica pelo origem de cadastro
    ORDER BY origem_cadastro --ordena pela origem de cadastro tb
"""))
#Este GROUP BY é conferência, não construção. A tabela já está escrita; o agrupamento aqui só serve para olhar o resultado. A proibição vale para o que é materializado na silver.

In [0]:
#só 20 das 729 empresas nacionais têm código icao --> o cadastro é dominado por aviação agrícola, táxi aéreo e aeroclube, que não têm código de três letras. Quem voa linha regular tem. Isso volta no marco-07, quando o join com o VRA for medido.

#silver.aerodromos e silver.codigos_operacao --> espelhos 
# Uma tabela de referência para cada uma do bronze, tipada e documentada. Duas coisas valem comentário:

#altitude vem como "193,0" — vírgula decimal. Vira double com um replace, mudando "," por um "."

#A coluna que o cabeçalho chama de UF contém "Acre", "São Paulo": é o nome da unidade federativa por extenso, não a sigla. Quem escrever WHERE uf = 'SP' recebe zero linhas e vai achar que o dado sumiu. O nome da coluna passa a dizer a verdade (uf_nome) e o COMMENT avisa. Renomear e documentar é governança; inventar a sigla seria transformação de negócio.

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.aerodromos AS
SELECT
  icao,
  ciad,
  nome,
  municipio,
  uf                                            AS uf_nome,
  municipio_servido,
  uf_servido                                    AS uf_servido_nome,
  latitude                                      AS latitude_dms,
  longitude                                     AS longitude_dms,
  try_cast(replace(altitude, ',', '.') AS DOUBLE) AS altitude_m, --modificação da tipagem da coluna "altitude"
  situacao,
  _ingerido_em,
  current_timestamp()                           AS _transformado_em
FROM voebem.bronze.aerodromos
""")
#vamos buscar abaixo umas infos que precisamos pra codigo de operaçao dos voos
spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.codigos_operacao AS
SELECT
  dominio,
  codigo,
  descricao,
  current_timestamp() AS _transformado_em
FROM voebem.bronze.codigos_operacao
""")
#união com aerodromos e codigos de operação de voos. 
display(spark.sql("""
    SELECT 'aerodromos' AS tabela,
           (SELECT COUNT(*) FROM voebem.bronze.aerodromos) AS bronze,
           (SELECT COUNT(*) FROM voebem.silver.aerodromos) AS silver
    UNION ALL
    SELECT 'codigos_operacao',
           (SELECT COUNT(*) FROM voebem.bronze.codigos_operacao),
           (SELECT COUNT(*) FROM voebem.silver.codigos_operacao)
"""))
#aerodromos sao aeroportos. fazemos a conexão de entrada de voos, ou seja, quando voos aconteceram para descobrirmos origem e destino, e aí correlacionar para ver se tem uma rota sobrecarregada, ou um destino que tem problemas meteorologicos de forma constante, etc


In [0]:
#bronze e silver possuem o mesmo número, então tá certo. a conferência mostrou que tá correto

#Metadados: documentação não é enfeite: o consumidor final deste pipeline é um LLM, e o COMMENT é literalmente o que ele lê para decidir qual coluna usar. Coluna sem comentário é coluna que a IA vai usar errado.

#O comentário descreve significado de negócio, não tipo de dado. "TIMESTAMP da partida" não ajuda ninguém; "horário em que a aeronave efetivamente saiu do solo" ajuda.
COMENTARIOS_VRA = {
    "icao_empresa":            "Codigo ICAO de tres letras da empresa aerea que operou a etapa. Chave para silver.empresas.",
    "numero_voo":              "Numero do voo divulgado pela companhia. Identificador comercial, nao numerico: pode ter zero a esquerda e se repete entre datas.",
    "codigo_di":               "Codigo de autorizacao (DI) da etapa: distingue etapa regular, extra, de retorno, charter. Descricao em silver.codigos_operacao (dominio codigo_di).",
    "codigo_tipo_linha":       "Codigo do tipo de linha: N e C domesticas, I e G internacionais. Descricao em silver.codigos_operacao (dominio codigo_tipo_linha).",
    "icao_origem":             "Codigo ICAO do aerodromo de onde a etapa partiu. Chave para silver.aerodromos - aeroportos estrangeiros nao constam no cadastro da ANAC.",
    "icao_destino":            "Codigo ICAO do aerodromo onde a etapa pousou. Mesma observacao de cobertura da origem.",
    "partida_prevista":        "Horario de partida programado pela companhia, na hora local do aeroporto de origem.",
    "partida_prevista_data":   "Data da partida programada, separada para facilitar analise por dia.",
    "partida_prevista_hora":   "Hora e minuto da partida programada (HH:mm), separada para analise por faixa horaria.",
    "partida_real":            "Horario em que a aeronave efetivamente saiu. Nulo em voo cancelado, que nao chegou a partir.",
    "partida_real_data":       "Data da partida efetiva.",
    "partida_real_hora":       "Hora e minuto da partida efetiva (HH:mm).",
    "chegada_prevista":        "Horario de chegada programado, na hora local do aeroporto de destino.",
    "chegada_prevista_data":   "Data da chegada programada.",
    "chegada_prevista_hora":   "Hora e minuto da chegada programada (HH:mm).",
    "chegada_real":            "Horario em que a aeronave efetivamente pousou. Nulo em voo cancelado.",
    "chegada_real_data":       "Data da chegada efetiva.",
    "chegada_real_hora":       "Hora e minuto da chegada efetiva (HH:mm).",
    "situacao_voo":            "Situacao informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando nao.",
    "codigo_justificativa":    "Motivo declarado do atraso. Deixou de ser exigido pela ANAC em abril de 2020 com a revogacao da IAC 1504: vem vazio em toda a janela deste projeto.",
    "atraso_partida_min":      "Minutos entre a partida programada e a partida efetiva. Positivo e atraso, negativo e antecipacao. Aritmetica pura: nao aplica limiar de pontualidade.",
    "atraso_chegada_min":      "Minutos entre a chegada programada e a chegada efetiva. Positivo e atraso, negativo e antecipacao.",
    "minutos_recuperados":     "Minutos que a etapa recuperou em voo: atraso de partida menos atraso de chegada. Positivo significa que chegou menos atrasada do que saiu.",
    "_arquivo_origem":         "Auditoria: nome do arquivo CSV mensal da ANAC de onde a linha veio.",
    "_ingerido_em":            "Auditoria: momento em que a linha entrou no bronze.",
    "_transformado_em":        "Auditoria: momento em que a silver foi reconstruida a partir do bronze.",
}
#um comentário para cada coluna:
for coluna, comentario in COMENTARIOS_VRA.items():
    spark.sql(f"ALTER TABLE voebem.silver.vra ALTER COLUMN {coluna} COMMENT '{comentario}'")

print(f"{len(COMENTARIOS_VRA)} colunas comentadas em silver.vra")
#dando mais contexto, pensando que podemos utilizar isso com um agente e para pessoas de diferentes áreas possam entender informações da table e tratamento dos dados.
#ter um catálogo de dados é um bom ato de governança
#pra achar um dado, nem sempre vou saber o código dele, mas se bater com descrição, encontro qual estou procurando
#lembrando que engenharia de dados tem muita coisa relacionada a parte técnica, mas é MUITO sobre entender das regras de negócios e da empresa.